# 🌫️ C2P-Net: AI Single Image Dehazing Web Studio (CVPR 2023)
Run C2P-Net single image dehazing on a **Free NVIDIA T4 GPU** with an instant public Web UI (Zero local Mac lag!).

### Step 1: Check GPU Acceleration

In [ ]:
# Ensure GPU is enabled (Runtime > Change runtime type > T4 GPU)
!nvidia-smi

### Step 2: Clone Repository & Install Dependencies

In [ ]:
%cd /content
!rm -rf C2P-NET
!git clone https://github.com/hemantmeena2005/C2P-NET.git
%cd /content/C2P-NET

!pip install -q gradio torchvision pillow

### Step 3: Launch Interactive Web Studio on GPU (Public Web Link)

In [ ]:
import os, time, torch
import torchvision.transforms as tfs
from PIL import Image
import gradio as gr
from models.C2PNet import C2PNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[*] Active Hardware Acceleration: {device}")

# Pre-load models into GPU VRAM for instant sub-second response
MODELS = {
    'Indoor Scene (ITS - 42.56 dB)': 'trained_models/ITS.pkl',
    'Outdoor Scene (OTS - 36.68 dB)': 'trained_models/OTS.pkl'
}

loaded_nets = {}

def get_net(model_key):
    if model_key in loaded_nets:
        return loaded_nets[model_key]
    
    path = MODELS[model_key]
    print(f"[*] Loading checkpoint from {path} onto {device}...")
    net = C2PNet(gps=3, blocks=19).to(device)
    ckp = torch.load(path, map_location=device, weights_only=False)
    state_dict = ckp['model'] if 'model' in ckp else ckp
    net.load_state_dict(state_dict)
    net.eval()
    loaded_nets[model_key] = net
    return net

# Preload default indoor model
if os.path.exists(MODELS['Indoor Scene (ITS - 42.56 dB)']):
    get_net('Indoor Scene (ITS - 42.56 dB)')

def dehaze_inference(input_img, model_choice):
    if input_img is None:
        return None, "No image provided"
    
    t0 = time.time()
    img = input_img.convert('RGB')
    orig_w, orig_h = img.size
    
    # Fast processing on GPU (auto-scale large 4K images to 1600p if needed)
    max_dim = 1600
    if max(orig_w, orig_h) > max_dim:
        ratio = max_dim / max(orig_w, orig_h)
        img = img.resize((int(orig_w * ratio), int(orig_h * ratio)), Image.Resampling.LANCZOS)
    
    net = get_net(model_choice)
    img_t = tfs.ToTensor()(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        pred = net(img_t)
    
    pred = pred.clamp(0, 1).squeeze(0).cpu()
    out_pil = tfs.ToPILImage()(pred)
    
    dt = round(time.time() - t0, 3)
    stats = f"⚡ Processed in {dt}s on {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'} | Resolution: {orig_w}x{orig_h}"
    return out_pil, stats

# Gradio Interface with Split Viewer
with gr.Blocks(title="C2P-Net Dehazer Studio", theme=gr.themes.Soft(primary_hue="cyan")) as demo:
    gr.Markdown(
        """
        # 🌫️ C2P-Net: Physics-Aware Single Image Dehazing
        ### *CVPR 2023 - Curricular Contrastive Regularization for Physics-aware Single Image Dehazing*
        Lightning-fast GPU image dehazing. Upload any hazy photo to restore full crystal-clear clarity.
        """
    )
    
    with gr.Row():
        with gr.Column():
            input_box = gr.Image(type="pil", label="1. Input Hazy Image")
            model_select = gr.Dropdown(
                choices=list(MODELS.keys()),
                value='Indoor Scene (ITS - 42.56 dB)',
                label="2. Scene Checkpoint Model"
            )
            run_btn = gr.Button("✨ Enhance & Dehaze Image", variant="primary", size="lg")
            
        with gr.Column():
            output_box = gr.Image(type="pil", label="3. Restored Clear Image")
            status_text = gr.Markdown("Ready to dehaze.")
            
    run_btn.click(
        fn=dehaze_inference,
        inputs=[input_box, model_select],
        outputs=[output_box, status_text]
    )

# share=True provides an instant, secure public link to access anywhere
demo.launch(share=True, debug=True)